In [3]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

In [4]:
sentences = [
    "i love deep learning",
    "i love machine learning",
    "deep learning is powerful",
    "machine learning is fun",
    "i love python",
    "python is powerful"
]

print(sentences)

['i love deep learning', 'i love machine learning', 'deep learning is powerful', 'machine learning is fun', 'i love python', 'python is powerful']


In [5]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(sentences)

In [6]:
word_index = tokenizer.word_index
total_words = len(word_index) + 1

print("Word index:", word_index)
print("Total words:", total_words)

Word index: {'learning': 1, 'i': 2, 'love': 3, 'is': 4, 'deep': 5, 'machine': 6, 'powerful': 7, 'python': 8, 'fun': 9}
Total words: 10


In [7]:
sample = "i love deep learning"
token_list = tokenizer.texts_to_sequences([sample])[0]

print("Sentence:", sample)
print("Token list:", token_list)

Sentence: i love deep learning
Token list: [2, 3, 5, 1]


In [8]:
input_sequences = []

for sentence in sentences:
    token_list = tokenizer.texts_to_sequences([sentence])[0]
    for i in range(1, len(token_list)):
        seq = token_list[:i+1]
        input_sequences.append(seq)

print(input_sequences)

[[2, 3], [2, 3, 5], [2, 3, 5, 1], [2, 3], [2, 3, 6], [2, 3, 6, 1], [5, 1], [5, 1, 4], [5, 1, 4, 7], [6, 1], [6, 1, 4], [6, 1, 4, 9], [2, 3], [2, 3, 8], [8, 4], [8, 4, 7]]


In [9]:
max_len = max(len(seq) for seq in input_sequences)
print("Max sequence length:", max_len)

Max sequence length: 4


In [10]:
padded_sequences = pad_sequences(input_sequences, maxlen=max_len, padding='pre')

print(padded_sequences)

[[0 0 2 3]
 [0 2 3 5]
 [2 3 5 1]
 [0 0 2 3]
 [0 2 3 6]
 [2 3 6 1]
 [0 0 5 1]
 [0 5 1 4]
 [5 1 4 7]
 [0 0 6 1]
 [0 6 1 4]
 [6 1 4 9]
 [0 0 2 3]
 [0 2 3 8]
 [0 0 8 4]
 [0 8 4 7]]


In [11]:
X = padded_sequences[:, :-1]
y = padded_sequences[:, -1]

print("X:\n", X)
print("y:\n", y)

X:
 [[0 0 2]
 [0 2 3]
 [2 3 5]
 [0 0 2]
 [0 2 3]
 [2 3 6]
 [0 0 5]
 [0 5 1]
 [5 1 4]
 [0 0 6]
 [0 6 1]
 [6 1 4]
 [0 0 2]
 [0 2 3]
 [0 0 8]
 [0 8 4]]
y:
 [3 5 1 3 6 1 1 4 7 1 4 9 3 8 4 7]


In [12]:
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (16, 3)
y shape: (16,)


In [13]:
model = Sequential()
model.add(Embedding(total_words, 10, input_length=max_len-1))
model.add(LSTM(20))
model.add(Dense(total_words, activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [14]:
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [15]:
model.fit(X, y, epochs=200, verbose=1)

Epoch 1/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 6s 6s/step - accuracy: 0.0625 - loss: 2.3043
Epoch 2/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.0625 - loss: 2.3019
Epoch 3/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.0000e+00 - loss: 2.2995
Epoch 4/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.0625 - loss: 2.2972
Epoch 5/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.0625 - loss: 2.2948
Epoch 6/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.1250 - loss: 2.2924
Epoch 7/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.1875 - loss: 2.2901
Epoch 8/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.2500 - loss: 2.2877
Epoch 9/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.2500 - loss: 2.2852
Epoch 10/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - accuracy: 0.3125 - loss: 2.2828
Epoch 11/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.2500 - loss: 2.2803
Epoch 12/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.3125 - 

In [16]:
def predict_next_word(text):
    token_list = tokenizer.texts_to_sequences([text])[0]
    token_list = pad_sequences([token_list], maxlen=max_len-1, padding='pre')

    predicted = model.predict(token_list, verbose=0)
    predicted_index = np.argmax(predicted)

    for word, index in word_index.items():
        if index == predicted_index:
            return word


In [17]:
print("i love ->", predict_next_word("i love"))
print("deep learning ->", predict_next_word("deep learning"))
print("python is ->", predict_next_word("python is"))

i love -> deep
deep learning -> is
python is -> powerful


In [18]:
user_text = input("Enter your text: ")
print("Next word:", predict_next_word(user_text))

Enter your text: hello
Next word: learning
